# PaperPilot v3 — LoRA Fine-tuning

**Fix:** Training targets had `@cite_N` tokens as dominant signal → model learned to output only citations.
**Solution:** Strip `@cite_N` from targets before training, so the model learns prose synthesis.

## Steps
1. Install deps
2. Load `multi_x_science_sum` dataset
3. Clean targets (remove `@cite_N`)
4. Build prompt in the same format as inference
5. LoRA fine-tune Qwen2.5-0.5B-Instruct
6. Evaluate on held-out examples
7. Save adapter

**Runtime:** T4 GPU required. ~45–90 min for 3 epochs.

In [ ]:
# ── 1. Install ──────────────────────────────────────────────
!pip install -q transformers peft datasets accelerate bitsandbytes

In [ ]:
# ── 2. Config ────────────────────────────────────────────────
import re, json, os

MODEL_ID   = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = '/content/paperpilot_lora_v3'
MAX_SEQ    = 1024          # fits T4 comfortably
MAX_REFS   = 4             # papers per example
MIN_TARGET_WORDS = 30      # discard targets shorter than this
EPOCHS     = 3
BATCH      = 2
GRAD_ACCUM = 8             # effective batch = 16
LR         = 2e-4
LORA_R     = 16            # doubled from v1 (was 8)
LORA_ALPHA = 32

print('Config OK')

In [ ]:
# ── 3. Load dataset ──────────────────────────────────────────
# multi_x_science_sum is now available as standard Parquet on HuggingFace.
# No trust_remote_code needed.
from datasets import load_dataset

ds = load_dataset('yaolu/multi_x_science_sum')
print(ds)
print('Train sample keys:', list(ds['train'][0].keys()))
print('First row preview:')
row = ds['train'][0]
print(' abstract:', row.get('abstract','')[:200])
print(' related_work:', row.get('related_work','')[:200])
print(' ref_abstract type:', type(row.get('ref_abstract')))

In [ ]:
# ── 4. Cleaning helpers ──────────────────────────────────────
CITE_PAT = re.compile(r'@cite_\d+', re.IGNORECASE)

def strip_citations(text: str) -> str:
    """Remove @cite_N tokens; collapse whitespace."""
    text = CITE_PAT.sub('', text)
    text = re.sub(r'\s{2,}', ' ', text).strip()
    # Remove dangling punctuation left by stripped cites e.g. '[ , ]'
    text = re.sub(r'\[\s*[,;\s]*\]', '', text)
    text = re.sub(r'\s+([.,;])', r'\1', text)
    return text.strip()

def build_prompt(query: str, refs: list, titles: list = None) -> str:
    """Matches inference prompt exactly (backend/services/llm_service.py _build_prompt)."""
    ref_parts = []
    for i, r in enumerate(refs):
        header = f'Reference Paper {i+1}'
        if titles and i < len(titles):
            header += f' ({titles[i]})'
        ref_parts.append(f'{header}:\n{r[:800]}')
    refs_text = '\n\n'.join(ref_parts)
    return (
        'Write a related-work section that synthesizes the following reference papers '
        'in relation to the query paper. Only make claims supported by the supplied abstracts. '
        'Cite each reference paper using its number in brackets. '
        'Do not invent results, comparisons, or citations.\n\n'
        f'Query Paper Abstract:\n{query[:600]}\n\n'
        f'Reference Papers:\n{refs_text}'
    )

print('Helpers OK')

In [ ]:
# ── 5. Build training examples ───────────────────────────────
# Field names in allenai/multi_x_science_sum:
#   aid, abstract, related_work, ref_abstract (dict with 'abstract' list)
def process_split(split_data, max_examples=None):
    examples = []
    for row in split_data:
        query   = (row.get('abstract') or '').strip()
        related = (row.get('related_work') or '').strip()

        # ref_abstract is a dict with key 'abstract' containing a list of strings
        ref_field = row.get('ref_abstract', {})
        if isinstance(ref_field, dict):
            refs_raw = ref_field.get('abstract', [])
        elif isinstance(ref_field, list):
            refs_raw = ref_field
        else:
            refs_raw = []

        if not query or not related or not refs_raw:
            continue

        # Strip @cite_N from query and target
        query_clean   = strip_citations(query)
        target_clean  = strip_citations(related)

        # Skip weak targets
        if len(target_clean.split()) < MIN_TARGET_WORDS:
            continue

        # Pick at most MAX_REFS non-empty reference abstracts
        refs = [strip_citations(r) for r in refs_raw if r and len(str(r).strip()) > 40]
        refs = refs[:MAX_REFS]
        if not refs:
            continue

        prompt = build_prompt(query_clean, refs)
        examples.append({'prompt': prompt, 'target': target_clean})
        if max_examples and len(examples) >= max_examples:
            break

    return examples

train_ex = process_split(ds['train'],      max_examples=8000)
val_ex   = process_split(ds['validation'], max_examples=500)
test_ex  = process_split(ds['test'],       max_examples=100)

print(f'Train: {len(train_ex)} | Val: {len(val_ex)} | Test: {len(test_ex)}')
print('\n--- Sample prompt (first 400 chars):')
print(train_ex[0]['prompt'][:400])
print('\n--- Sample target (first 300 chars):')
print(train_ex[0]['target'][:300])

In [ ]:
# ── 6. Verify NO @cite_N in any target ───────────────────────
cite_in_targets = sum(1 for e in train_ex if '@cite_' in e['target'])
print(f'@cite_N in targets: {cite_in_targets}/{len(train_ex)} (must be 0)')
assert cite_in_targets == 0, 'Still have @cite_N in targets!'

In [ ]:
# ── 7. Load tokenizer + model ────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.config.use_cache = False

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# ── 8. Tokenise ──────────────────────────────────────────────
from datasets import Dataset

def tokenize(ex):
    messages = [
        {'role': 'system', 'content': 'You are a scientific writing assistant.'},
        {'role': 'user',   'content': ex['prompt']},
    ]
    prompt_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True
    )
    target_ids = tokenizer(
        ex['target'], add_special_tokens=False
    )['input_ids'] + [tokenizer.eos_token_id]

    combined = (prompt_ids + target_ids)[:MAX_SEQ]
    labels   = ([-100] * len(prompt_ids) + target_ids)[:MAX_SEQ]
    return {
        'input_ids':      combined,
        'attention_mask': [1] * len(combined),
        'labels':         labels,
    }

train_ds = Dataset.from_list([tokenize(e) for e in train_ex])
val_ds   = Dataset.from_list([tokenize(e) for e in val_ex])
print(f'Tokenised — train: {len(train_ds)}, val: {len(val_ds)}')

In [ ]:
# ── 9. Train ─────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=True,
    gradient_checkpointing=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=50,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer, padding=True, label_pad_token_id=-100
    ),
)
trainer.train()

In [ ]:
# ── 10. Save adapter ─────────────────────────────────────────
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Adapter saved to', OUTPUT_DIR)

In [ ]:
# ── 11. Evaluate on held-out test examples ───────────────────
import re as _re

model.eval()

def generate(prompt: str, max_new_tokens: int = 400) -> str:
    messages = [
        {'role': 'system', 'content': 'You are a scientific writing assistant.'},
        {'role': 'user',   'content': prompt},
    ]
    enc = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    )
    if hasattr(enc, 'input_ids'):
        ids = enc.input_ids
    else:
        ids = enc
    ids = ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens,
                             do_sample=False,
                             pad_token_id=tokenizer.pad_token_id,
                             eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()

results = []
for ex in test_ex[:5]:
    pred = generate(ex['prompt'])
    cite_count = len(_re.findall(r'@cite_\d+', pred))
    words = len(pred.split())
    results.append({'words': words, 'cite_spam': cite_count, 'preview': pred[:200]})
    print(f'words={words} | @cite spam={cite_count}')
    print('  ', pred[:200])
    print()

avg_words = sum(r['words'] for r in results) / len(results)
avg_spam  = sum(r['cite_spam'] for r in results) / len(results)
print(f'\nAvg words: {avg_words:.0f} | Avg @cite spam: {avg_spam:.1f}')
print('Pass if avg_words > 40 AND avg_spam == 0')

In [ ]:
# ── 12. Zip and download ─────────────────────────────────────
import shutil
shutil.make_archive('/content/paperpilot_lora_v3', 'zip', OUTPUT_DIR)
from google.colab import files
files.download('/content/paperpilot_lora_v3.zip')
print('Download started')